## Installations and Imports

In [ ]:
!pip install openai-agents dspy
# INITIALIZE THE API HERE
# AZURE_API_KEY     = "FILL THE RESPECTIVE VALUE HERE"
# AZURE_ENDPOINT    = "FILL THE RESPECTIVE VALUE HERE"
# AZURE_API_VERSION = "FILL THE RESPECTIVE VALUE HERE"
# AZURE_DEPLOYMENT  = "FILL THE RESPECTIVE VALUE HERE"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.2
    Uninstalling typeguard-4.5.2:
      Successfully uninstalled typeguard-4.5.2


## Email Extraction

In [ ]:
# Importing emails
import json
emails = dict()
with open("emails_new.json", encoding="utf-8") as f:
    emails = json.load(f)
for key, body in emails.items():
    print(key, "→", body[:200])


email_01 → FROM: procurement@delta-infra.com
TO: bids@acmecorp.com
SUBJECT: Bid Request – Industrial Welding Robots

Hi,

We are looking to procure 15 units of 6-axis industrial welding robots for our new assemb
email_02 → FROM: supply@greenfield-auto.com
TO: vendors@acmemotors.com
SUBJECT: RFQ – EV Battery Packs (Q3 2026)

Dear Vendor,

We require 2,000 units of 62 kWh LFP battery packs and 800 units of 88 kWh LFP batt
email_03 → FROM: it.buying@novatechsols.com
TO: sales@acmecorp.com
SUBJECT: Bulk Laptop Procurement – Bid Invitation

Hello,

We are expanding our engineering team and need 120 laptops urgently. Specs required: 
email_04 → FROM: ops@metro-construct.co.uk
TO: procurement.bids@acmecorp.com
SUBJECT: Request for Quotation – OLED Display Panels

Hi Team,

We need 50,000 units of 6.7 inch AMOLED panels (120Hz, 2400x1080) for 
email_05 → FROM: admin@sunrise-pharma.in
TO: supply.bids@acmecorp.com
SUBJECT: Bid Invitation – Office Furniture and Workstations

Dear Sir/Madam,

Sunr

## Agent-Agent With DSPy - General Usecase

In [ ]:
import dspy
from openai import AsyncAzureOpenAI
from agents import set_default_openai_client, set_default_openai_api
import asyncio
from dataclasses import dataclass
from agents import Agent, Runner, ModelSettings, set_tracing_disabled
set_tracing_disabled(True)


client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model       = f"azure/{AZURE_DEPLOYMENT}",
    api_key     = AZURE_API_KEY,
    api_base    = AZURE_ENDPOINT,
    api_version = AZURE_API_VERSION,
    max_tokens  = 1024,
    temperature = 0.0,   # ← deterministic skeleton generation
)
dspy.configure(lm=lm)

# ── DSPy: raw prompt → JSON skeleton with empty values ───────────────────────

class BuildSkeleton(dspy.Signature):
    """
    Given a user request, produce a JSON skeleton.
    Rules:
      - Keys must be short, descriptive snake_case strings.
      - Fill ONLY the keys you can infer directly from the request
        (e.g. name, category, type). Leave everything else as "".
      - Use plenty of keys covering: identity, geography, economy,
        history, culture, politics, notable_facts, and any topic-specific fields.
      - Output raw JSON only — no markdown fences, no explanation.
        make sure you provide all possible keys as possible, but make sure it should neither lengthy json not short json, it should address all aspects of the request.
        arrange the keys in catogories wise, like finance, history, specifications, desclaimers, hardware_requests etc.
    """
    user_request:  str = dspy.InputField(desc="The raw user request, e.g. 'Tell me about China'")
    json_skeleton: str = dspy.OutputField(desc="Raw JSON object with snake_case keys and empty string values, except for fields directly inferable from the request")

def dspy_transform(prompt: str) -> str:
    return dspy.Predict(BuildSkeleton)(user_request=prompt).json_skeleton

# ── SubAgent: receives skeleton → fills every value ──────────────────────────

sub_agent = Agent(
    name="SubAgent",
    model=AZURE_DEPLOYMENT,
    model_settings=ModelSettings(
        max_tokens  = 2048,
        temperature = 0.0,   # ← fixed typo: was 'temparature'
    ),
    instructions=(
        "You are a data enrichment agent. "
        "You receive a JSON object where most values are empty strings. "
        "Fill in EVERY empty value with accurate, factual information. "
        "Do not remove any keys. Do not add prose. "
        "Return only the completed JSON — no markdown fences, no explanation."
    ),
)

# ── Pipeline ──────────────────────────────────────────────────────────────────

async def run_pipeline(user_prompt: str) -> str:
    print(f"[User]       {user_prompt}\n")

    # Step 1 — DSPy builds the skeleton
    skeleton = dspy_transform(user_prompt.strip())
    print(f"[DSPy]       {skeleton}\n")

    # Step 2 — SubAgent fills every value
    result = await Runner.run(sub_agent, input=skeleton)
    print(f"[SubAgent]   {result.final_output}")

    return result.final_output

await run_pipeline("Write about china")

[User]       Write about china

[DSPy]       {
  "identity": {
    "name": "China",
    "official_name": "People's Republic of China",
    "type": "country",
    "region": "East Asia"
  },
  "geography": {
    "area": "",
    "population": "",
    "capital": "Beijing",
    "major_cities": "",
    "climate": "",
    "landmarks": "",
    "borders": ""
  },
  "economy": {
    "currency": "Renminbi (Yuan)",
    "gdp": "",
    "major_industries": "",
    "trade_partners": "",
    "economic_system": "Socialist market economy"
  },
  "history": {
    "ancient_history": "",
    "modern_history": "",
    "dynasties": "",
    "key_events": "",
    "revolution": "",
    "founding_date": "October 1, 1949"
  },
  "culture": {
    "language": "Mandarin Chinese",
    "religions": "",
    "traditions": "",
    "festivals": "",
    "cuisine": "",
    "arts": "",
    "sports": ""
  },
  "politics": {
    "government_type": "Communist one-party state",
    "current_leader": "",
    "political_parties": "

'{\n  "identity": {\n    "name": "China",\n    "official_name": "People\'s Republic of China",\n    "type": "country",\n    "region": "East Asia"\n  },\n  "geography": {\n    "area": "9,596,961 square kilometers",\n    "population": "1,425,893,465 (2023 estimate)",\n    "capital": "Beijing",\n    "major_cities": "Shanghai, Guangzhou, Shenzhen, Chongqing, Tianjin",\n    "climate": "Varies from subarctic in the north to tropical in the south",\n    "landmarks": "Great Wall of China, Forbidden City, Terracotta Army, Potala Palace, Zhangjiajie National Forest Park",\n    "borders": "14 countries: Afghanistan, Bhutan, India, Kazakhstan, Kyrgyzstan, Laos, Mongolia, Myanmar, Nepal, North Korea, Pakistan, Russia, Tajikistan, Vietnam"\n  },\n  "economy": {\n    "currency": "Renminbi (Yuan)",\n    "gdp": "18.32 trillion USD (2022)",\n    "major_industries": "Manufacturing, technology, agriculture, mining, textiles, electronics",\n    "trade_partners": "United States, European Union, Japan, South

## Agent-Agent With DSPy - Email case

In [34]:
for key in emails.keys():
  print("="*60)
  print(emails.get(key))

FROM: procurement@delta-infra.com
TO: bids@acmecorp.com
SUBJECT: Bid Request – Industrial Welding Robots

Hi,

We are looking to procure 15 units of 6-axis industrial welding robots for our new assembly line in Pune. Our budget is capped at ₹2.2 crore for the entire lot. We need delivery completed by August 15, 2026. Brands we are open to: KUKA, ABB, or equivalent. Please confirm if you can match this requirement and submit your best bid by June 10, 2026. Include warranty terms and AMC pricing in your response.

Regards,
Sandeep Rao
Procurement Head | Delta Infrastructure
FROM: supply@greenfield-auto.com
TO: vendors@acmemotors.com
SUBJECT: RFQ – EV Battery Packs (Q3 2026)

Dear Vendor,

We require 2,000 units of 62 kWh LFP battery packs and 800 units of 88 kWh LFP battery packs for our Q3 production run. Total budget allocated is €18 million for the combined lot. Delivery must be staggered — first 1,200 units by September 1, 2026 and the remaining by October 20, 2026. All units must be